# Passo 1: Monta Google Drive + leggi secret Roboflow senza hard-code

In [2]:
from google.colab import drive, userdata
drive.mount('/content/drive')
RUNS_DIR = '/content/drive/MyDrive/basketball_analysis/basketball_analysis/last_runs'

Mounted at /content/drive


## Suggerimento: mantieni attiva la sessione
- Evita lo stop del PC mentre Colab addestra (sospensione e blocco schermo disattivati).
- In caso di interruzioni impreviste, i checkpoint su Drive permettono di riprendere con la cella *Riprendere il training*.


# Passo 2: Naviga nella Cartella del Progetto

In [3]:
import os, sys, torch, platform
PROJECT_PATH = '/content/drive/MyDrive/basketball_analysis/basketball_analysis'
os.makedirs(PROJECT_PATH, exist_ok=True)
os.chdir(PROJECT_PATH)
!pwd
!ls -la

/content/drive/MyDrive/basketball_analysis/basketball_analysis
total 165388
drwx------ 2 root root     4096 Aug  6 17:38 ball_acquisition
drwx------ 2 root root     4096 Aug  7 13:44 basketball-court-detection-2-1
drwx------ 2 root root     4096 Aug 17 20:22 basketball-player-detection-2-1
drwx------ 2 root root     4096 Aug  6 17:38 configs
drwx------ 2 root root     4096 Aug  6 17:38 court_keypoint_detector
drwx------ 2 root root     4096 Sep  2 17:17 datasets_ball_only
-rw------- 1 root root      448 Jul 28 14:14 Dockerfile
drwx------ 2 root root     4096 Aug  6 17:38 drawers
-rw------- 1 root root     2388 Aug  1 14:44 extract_frames.py
-rw------- 1 root root     8838 Jul 28 16:25 GEMINI.md
-rw------- 1 root root     3310 Jul 31 14:59 generate_stubs.py
drwx------ 2 root root     4096 Aug  6 17:38 .git
-rw------- 1 root root       35 Aug  1 14:47 .gitignore
drwx------ 2 root root     4096 Aug  6 17:38 images
drwx------ 2 root root     4096 Aug  6 17:38 input_videos
drwx------ 2 root

Dovresti vedere l'elenco dei file del tuo progetto (`main.py`, `models/`, `input_videos/`, ecc.).


# Info ambiente / GPU

In [4]:
print("Python:", platform.python_version())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.11
CUDA available: True
GPU: NVIDIA L4


Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [5]:
!nvidia-smi

Thu Sep  4 09:01:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   47C    P8             17W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Passo 3: Installa le Librerie Necessarie (versioni bloccate)

In [6]:
!pip -q install ultralytics roboflow pandas matplotlib

import yaml, json, shutil, re
from pathlib import Path
from ultralytics import YOLO
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 44.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
import ultralytics, sys
print("ultralytics:", ultralytics.__version__, "| python:", sys.version)

ultralytics: 8.3.192 | python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]


In [8]:
# Imposta la cartella dei run/checkpoint su Drive (Ultralytics 8.3.x).
from ultralytics.utils import SETTINGS

# Usa la variabile definita prima
SETTINGS.update({'runs_dir': RUNS_DIR})

print("Ultralytics runs_dir:", SETTINGS.get('runs_dir', SETTINGS['runs_dir']))


Ultralytics runs_dir: /content/drive/MyDrive/basketball_analysis/basketball_analysis/last_runs


In [9]:
import os
from pathlib import Path
check = Path(SETTINGS.get('runs_dir', RUNS_DIR))
print("Esiste?", check.exists(), "| Percorso:", check)

Esiste? True | Percorso: /content/drive/MyDrive/basketball_analysis/basketball_analysis/last_runs


# Passo 4: Download del Dataset (senza hard-code della key)

In [10]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
assert ROBOFLOW_API_KEY

RF_WORKSPACE = "firstworkspace-riuxx"                         # <--- modifica se serve
RF_PROJECT   = "basketball-player-detection-2-68vvt"          # <--- modifica se serve
RF_VERSION   = 1                                              # <--- modifica se serve
RF_FORMAT    = "yolov8"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
version = project.version(RF_VERSION)
dataset = version.download(RF_FORMAT)

DATA_YAML = Path(dataset.location) / "data.yaml"
assert DATA_YAML.exists(), f"data.yaml non trovato in {dataset.location}"
print("Dataset scaricato in:", dataset.location)

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print("Classi originali:", data_cfg.get("names"))
print("Train:", data_cfg.get("train"))
print("Val:", data_cfg.get("val"))
print("Test:", data_cfg.get("test"))


loading Roboflow workspace...
loading Roboflow project...
Dataset scaricato in: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1
Classi originali: ['ball', 'number', 'player', 'player-dribble', 'player-fall', 'player-jump-shot', 'player-layup', 'player-screen', 'player-shot-block', 'referee', 'rim']
Train: ../train/images
Val: ../valid/images
Test: ../test/images


# Passo 5: Utility per percorsi e conteggi

In [13]:
from pathlib import Path
import yaml, os, sys

# Assicura che DATA_YAML punti al tuo data.yaml scaricato da Roboflow
# Se lo hai già in una variabile, lascialo così:
DATA_YAML = Path(DATA_YAML)  # deve esistere
assert DATA_YAML.exists(), f"data.yaml non trovato: {DATA_YAML}"

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

yaml_dir = DATA_YAML.parent.resolve()

# Candidati plausibili per la root del dataset:
candidates = [
    yaml_dir,                   # es. .../basketball-player-detection-2-1
    yaml_dir.parent,            # un livello sopra (in caso di path: ../)
    Path(data_cfg.get("path")).resolve() if data_cfg.get("path") else None
]
candidates = [c for c in candidates if c is not None]

def exists_pair(root: Path, split_name: str):
    # Gestisce val/valid
    cand_dirs = []
    for s in [split_name, "valid" if split_name=="val" else split_name]:
        cand_dirs.append(root / s / "images")
        cand_dirs.append(root / s / "labels")
    # Restituisce True se almeno una coppia images/labels esiste coerentemente
    ok = ( (root / split_name / "images").exists() and (root / split_name / "labels").exists() ) or \
         ( split_name=="val" and (root / "valid" / "images").exists() and (root / "valid" / "labels").exists() )
    return ok

def pick_split_root(root: Path):
    # Trova train + val|valid
    has_train = (root / "train" / "images").exists() and (root / "train" / "labels").exists()
    has_val   = (root / "val" / "images").exists()   and (root / "val" / "labels").exists()
    has_valid = (root / "valid" / "images").exists() and (root / "valid" / "labels").exists()
    return has_train and (has_val or has_valid)

dataset_root = None
for c in candidates:
    if pick_split_root(c):
        dataset_root = c
        break

# Se non trovato, cerca più a fondo nei candidati
if dataset_root is None:
    for c in candidates:
        for up in [c, c.parent]:
            # prova up/ (train|val|valid)/images
            if pick_split_root(up):
                dataset_root = up
                break
        if dataset_root is not None:
            break

assert dataset_root is not None, f"Impossibile individuare la root del dataset vicino a {yaml_dir}. Controlla la struttura."

# Determina se lo split di validazione si chiama 'val' o 'valid'
val_dir = "val" if (dataset_root / "val" / "images").exists() else "valid"
assert (dataset_root / val_dir / "images").exists(), f"Manca {val_dir}/images sotto {dataset_root}"
assert (dataset_root / "train" / "images").exists(), f"Manca train/images sotto {dataset_root}"

# Ricostruisci uno YAML 'pulito' con path assoluto e split semplici
sanitized = dict(data_cfg)  # copia
sanitized["path"]  = str(dataset_root.resolve())
sanitized["train"] = "train/images"
sanitized["val"]   = f"{val_dir}/images"
if (dataset_root / "test" / "images").exists():
    sanitized["test"] = "test/images"

# Scrivi un nuovo YAML affiancato
DATA_YAML_SAN = yaml_dir / "data_sanitized.yaml"
with open(DATA_YAML_SAN, "w") as f:
    yaml.safe_dump(sanitized, f, sort_keys=False)
print("Creato YAML sanificato:", DATA_YAML_SAN)

# D'ora in poi usa il sanificato
DATA_YAML = DATA_YAML_SAN

# Mostra check rapidi
print("\n[CHECK]")
print("path:", sanitized["path"])
print("train:", (Path(sanitized["path"]) / sanitized["train"]).resolve())
print("val:",   (Path(sanitized["path"]) / sanitized["val"]).resolve())
print("test:",  (Path(sanitized["path"]) / sanitized.get("test","-")).resolve() if "test" in sanitized else "-")

Creato YAML sanificato: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/data_sanitized.yaml

[CHECK]
path: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1
train: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/train/images
val: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/valid/images
test: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/test/images


In [14]:
from pathlib import Path
import yaml

DATA_YAML = Path(DATA_YAML)  # usa quello sanificato
with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

root = Path(data_cfg["path"]).resolve()

def resolve_split_dirs(split_key: str):
    # split_key: "train" / "val" / "test"
    rel = data_cfg.get(split_key)
    assert rel is not None, f"Chiave '{split_key}' mancante nello YAML."
    p = (root / rel).resolve()  # es. root/train/images
    if p.name.lower() == "images":
        imgs = p
        lbls = p.parent / "labels"
    else:
        imgs = p / "images"
        lbls = p / "labels"
    assert imgs.exists(), f"images non trovata: {imgs}"
    assert lbls.exists(), f"labels non trovata: {lbls}"
    return imgs, lbls

def count_label_files(lbl_dir: Path):
    return len(list(Path(lbl_dir).rglob("*.txt")))

tr_img, tr_lbl = resolve_split_dirs("train")
va_img, va_lbl = resolve_split_dirs("val")

print("Train images:", tr_img)
print("Train labels:", tr_lbl, f"(# {count_label_files(tr_lbl)})")
print("Val images:", va_img)
print("Val labels:", va_lbl,   f"(# {count_label_files(va_lbl)})")

# (Opzionale) test
try:
    te_img, te_lbl = resolve_split_dirs("test")
    print("Test images:", te_img)
    print("Test labels:", te_lbl, f"(# {count_label_files(te_lbl)})")
except AssertionError:
    print("Nessun test set definito (ok).")


Train images: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/train/images
Train labels: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/train/labels (# 3831)
Val images: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/valid/images
Val labels: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/valid/labels (# 253)
Test images: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/test/images
Test labels: /content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/test/labels (# 202)


# Passo 6 — (Opzionale) Split by video per evitare leakage

In [15]:
ENABLE_VIDEO_SPLIT = False  # metti True se vuoi ricreare split per video

import random
random.seed(42)

def extract_video_id(stem: str):
    # Personalizza questa regex al tuo naming.
    # Esempi: "gameA_frame_0001", "match12_000345"
    m = re.match(r"([a-zA-Z0-9]+)[-_]", stem)
    return m.group(1) if m else None

def rebuild_split_by_video(images_dir: Path, labels_dir: Path, out_root: Path,
                           val_ratio=0.15, test_ratio=0.0):
    out_root = Path(out_root)
    for sub in ["train/images","train/labels","val/images","val/labels","test/images","test/labels"]:
        (out_root / sub).mkdir(parents=True, exist_ok=True)
    # group by video id
    mapping = {}
    for img in images_dir.rglob("*.*"):
        vid = extract_video_id(img.stem)
        if not vid:
            continue
        mapping.setdefault(vid, []).append(img)
    vids = list(mapping.keys())
    if not vids:
        print("⚠️ Nessun video_id riconosciuto: skip ricostruzione split.")
        return None

    random.shuffle(vids)
    n = len(vids)
    n_val = max(1, int(n * val_ratio))
    n_test = max(0, int(n * test_ratio))

    val_vids = set(vids[:n_val])
    test_vids = set(vids[n_val:n_val+n_test])
    train_vids = set(vids[n_val+n_test:])

    def which_split(vid):
        if vid in val_vids: return "val"
        if vid in test_vids: return "test"
        return "train"

    moved = {"train":0,"val":0,"test":0}
    for vid, imgs in mapping.items():
        split = which_split(vid)
        for img in imgs:
            # copy img
            dst_img = out_root / f"{split}/images/{img.name}"
            shutil.copy2(img, dst_img)
            # copy matching label
            lbl = labels_dir / (img.stem + ".txt")
            if lbl.exists():
                dst_lbl = out_root / f"{split}/labels/{lbl.name}"
                shutil.copy2(lbl, dst_lbl)
                moved[split]+=1
    print("Fatti:", moved)
    # create yaml
    new_yaml = {
        "path": str(out_root),
        "train": str(out_root/"train"),
        "val": str(out_root/"val"),
        "test": str(out_root/"test"),
        "names": data_cfg.get("names")
    }
    out_yaml = out_root / "data_video_split.yaml"
    with open(out_yaml, "w") as f:
        yaml.safe_dump(new_yaml, f)
    return out_yaml

if ENABLE_VIDEO_SPLIT:
    OUT_SPLIT = Path("datasets_video_split")
    out_yaml = rebuild_split_by_video(tr_img.parent, tr_lbl.parent, OUT_SPLIT)
    if out_yaml:
        DATA_YAML = out_yaml
        with open(DATA_YAML, "r") as f:
            data_cfg = yaml.safe_load(f)
        print("Usando split-by-video:", DATA_YAML)

# Passo 7 — Crea dataset “ball-only” + YAML

In [16]:
from pathlib import Path
import yaml, shutil, os
import random

# Assumo che tu abbia già: tr_img, tr_lbl, va_img, va_lbl (dalla utility "sanificata")
BALL_CLASS_NAME = "ball"
names = data_cfg.get("names", [])
assert BALL_CLASS_NAME in names, f'"{BALL_CLASS_NAME}" non è tra le classi: {names}'
ball_id = names.index(BALL_CLASS_NAME)
print("ball_id:", ball_id)

# Dove creare il dataset ball-only (su disco locale, molto più veloce)
ball_root = Path("/content/datasets_ball_only")
(ball_root / "train/images").mkdir(parents=True, exist_ok=True)
(ball_root / "train/labels").mkdir(parents=True, exist_ok=True)
(ball_root / "val/images").mkdir(parents=True, exist_ok=True)
(ball_root / "val/labels").mkdir(parents=True, exist_ok=True)

# Prepara mapping STEMP → PATH immagine, solo dentro images_dir (niente ricerche globali)
def build_stem_map(images_dir: Path, exts=(".jpg",".jpeg",".png",".bmp",".JPG",".PNG")):
    m = {}
    for ext in exts:
        for p in images_dir.rglob(f"*{ext}"):
            m.setdefault(p.stem, p)
    return m

# Scrive un sotto-dataset copiando/collegando solo i frame che contengono "ball"
def build_ball_only(images_dir: Path, labels_dir: Path, out_dir: Path,
                    link_mode="link", include_negatives=1.0, seed=0, clear_out=False):
    """
    Crea un dataset monoclasse 'ball'.
    include_negatives: frazione di immagini senza palla da includere come negative (0..1).
    clear_out: se True, svuota 'images' e 'labels' di out_dir prima di ricreare.
    """
    import itertools

    random.seed(seed)
    img_out = out_dir / "images"
    lbl_out = out_dir / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    if clear_out:
        for p in itertools.chain(img_out.glob("*"), lbl_out.glob("*")):
            try:
                p.unlink()
            except Exception:
                pass

    stem_map = build_stem_map(images_dir)
    kept_pos = kept_neg = total = 0

    for txt in labels_dir.rglob("*.txt"):
        total += 1
        with open(txt, "r") as f:
            lines = [ln.strip() for ln in f if ln.strip()]

        # filtra solo la classe ball
        keep = []
        for ln in lines:
            p = ln.split()
            if p and p[0].isdigit() and int(float(p[0])) == ball_id:
                p[0] = "0"  # rimappa a single-class
                keep.append(" ".join(p))

        stem = txt.stem
        img_path = stem_map.get(stem)
        if not img_path:
            continue

        dst_img = img_out / img_path.name
        dst_lbl = lbl_out / f"{stem}.txt"

        if keep:
            with open(dst_lbl, "w") as w:
                w.write("\n".join(keep) + "\n")
            kept_pos += 1
        else:
            if include_negatives > 0 and random.random() < include_negatives:
                open(dst_lbl, "w").close()  # negativo esplicito
                kept_neg += 1
            else:
                continue

        # link -> hardlink -> copia
        if link_mode == "link":
            try:
                if dst_img.exists(): dst_img.unlink()
                os.symlink(img_path, dst_img)
            except Exception:
                try:
                    if dst_img.exists(): dst_img.unlink()
                    os.link(img_path, dst_img)
                except Exception:
                    shutil.copyfile(img_path, dst_img)
        else:
            shutil.copyfile(img_path, dst_img)

    print(f"kept_pos={kept_pos}, kept_neg={kept_neg}, total_labels={total}")
    return kept_pos + kept_neg, total

# Esegui velocemente su /content
k_tr, t_tr = build_ball_only(tr_img, tr_lbl, ball_root / "train",
                             link_mode="link", include_negatives=1.0, seed=42)
k_va, t_va = build_ball_only(va_img, va_lbl, ball_root / "val",
                             link_mode="link", include_negatives=1.0, seed=42)
print(f"train: kept {k_tr}/{t_tr} files | val: kept {k_va}/{t_va} files")

# Scrivi data.yaml per ball-only
BALL_DATA_YAML = ball_root / "data_ball_only.yaml"
with open(BALL_DATA_YAML, "w") as f:
    yaml.safe_dump({
        "path": str(ball_root),
        "train": "train",
        "val": "val",
        "names": ["ball"]
    }, f, sort_keys=False)
print("Creato:", BALL_DATA_YAML)

ball_id: 0


KeyboardInterrupt: 

# Passo 8 — Crea dataset “players semplificato” (player/referee/rim)

In [17]:
from pathlib import Path
import yaml, shutil, os

# 1) Mapping classi (resta invariato, ma lo lasciamo qui completo)
ORIG_NAMES = data_cfg.get("names", [])
print("Classi originali:", ORIG_NAMES)

def simplify_name(cls_name: str):
    n = cls_name.lower()
    if "ball" in n:
        return None  # escludi la palla da questo dataset
    if "player" in n:
        return "player"
    if "ref" in n or "umpire" in n:
        return "referee"
    if any(k in n for k in ["rim","hoop","basket","net","backboard"]):
        return "rim"
    # scarta tutto il resto (es. jersey number)
    return None

# costruisci mapping id_orig -> new_name
id_to_new = {}
for i, n in enumerate(ORIG_NAMES):
    newn = simplify_name(n)
    if newn is not None:
        id_to_new[i] = newn

# Ordine fisso delle nuove classi
ORDER = ["player","referee","rim"]
NEW_NAMES = [c for c in ORDER if c in set(id_to_new.values())]
name_to_newid = {n:i for i,n in enumerate(NEW_NAMES)}

print("Mappatura id_orig->nuovo:", {k: (ORIG_NAMES[k], id_to_new[k]) for k in id_to_new})
print("Nuove classi:", NEW_NAMES)

# 2) Utilità veloci per immagini/label
def build_stem_map(images_dir: Path, exts=(".jpg",".jpeg",".png",".bmp",".JPG",".PNG")):
    m = {}
    for ext in exts:
        for p in images_dir.rglob(f"*{ext}"):
            m.setdefault(p.stem, p)
    return m

def build_players_simplified(images_dir: Path, labels_dir: Path, out_dir: Path, link_mode="link"):
    """
    images_dir: .../train/images  (o .../val/images)
    labels_dir: .../train/labels  (o .../val/labels)
    out_dir:    destinazione (es. /content/datasets_players_simplified/train)
    link_mode:  "link" usa symlink (più veloce), "copy" copia i file
    """
    img_out = out_dir / "images"
    lbl_out = out_dir / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    stem_map = build_stem_map(images_dir)  # O(N) una sola volta
    kept = 0
    total = 0

    for txt in labels_dir.rglob("*.txt"):
        total += 1
        with open(txt, "r") as f:
            lines = [ln.strip() for ln in f if ln.strip()]

        keep = []
        for ln in lines:
            parts = ln.split()
            if len(parts) < 5 or not parts[0].isdigit():
                continue
            orig_id = int(float(parts[0]))
            if orig_id in id_to_new:
                new_id = name_to_newid[id_to_new[orig_id]]
                parts[0] = str(new_id)
                keep.append(" ".join(parts))

        stem = txt.stem
        img_path = stem_map.get(stem)
        if not img_path:
            continue

        dst_lbl = lbl_out / f"{stem}.txt"
        dst_img = img_out / img_path.name

        if keep:
            # POSITIVO: scrivi label rimappata
            with open(dst_lbl, "w") as w:
                w.write("\n".join(keep) + "\n")
        else:
            # NEGATIVO: .txt vuoto (utile per ridurre falsi positivi)
            open(dst_lbl, "w").close()

        # symlink -> fallback a copia
        if link_mode == "link":
            try:
                if dst_img.exists():
                    dst_img.unlink()
                os.symlink(img_path, dst_img)
            except Exception:
                shutil.copyfile(img_path, dst_img)
        else:
            shutil.copyfile(img_path, dst_img)

        kept += 1

    return kept, total

# 3) Costruisci dataset in /content (veloce)
players_root = Path("/content/datasets_players_simplified")
(train_kept, train_total) = build_players_simplified(tr_img, tr_lbl, players_root / "train", link_mode="link")
(val_kept, val_total)     = build_players_simplified(va_img, va_lbl, players_root / "val",   link_mode="link")
print(f"train: kept {train_kept}/{train_total} files | val: kept {val_kept}/{val_total} files")

# 4) YAML per il dataset semplificato
PLAYERS_DATA_YAML = players_root / "data_players_simplified.yaml"
with open(PLAYERS_DATA_YAML, "w") as f:
    yaml.safe_dump({"path": str(players_root), "train": "train", "val": "val", "names": NEW_NAMES}, f, sort_keys=False)
print("Creato:", PLAYERS_DATA_YAML)


Classi originali: ['ball', 'number', 'player', 'player-dribble', 'player-fall', 'player-jump-shot', 'player-layup', 'player-screen', 'player-shot-block', 'referee', 'rim']
Mappatura id_orig->nuovo: {2: ('player', 'player'), 3: ('player-dribble', 'player'), 4: ('player-fall', 'player'), 5: ('player-jump-shot', 'player'), 6: ('player-layup', 'player'), 7: ('player-screen', 'player'), 8: ('player-shot-block', 'player'), 9: ('referee', 'referee'), 10: ('rim', 'rim')}
Nuove classi: ['player', 'referee', 'rim']
train: kept 3831/3831 files | val: kept 253/253 files
Creato: /content/datasets_players_simplified/data_players_simplified.yaml


In [18]:
from pathlib import Path

def stats_labels(root, names):
    n = len(names)
    counts = [0]*n
    negatives = 0
    tot_imgs = 0
    for split in ["train","val"]:
        lbl_dir = Path(root)/split/"labels"
        for p in lbl_dir.glob("*.txt"):
            tot_imgs += 1
            lines = [ln.strip() for ln in open(p) if ln.strip()]
            if not lines:
                negatives += 1
                continue
            for ln in lines:
                cid = int(float(ln.split()[0]))
                assert 0 <= cid < n, f"ID fuori range in {p}: {cid}"
                counts[cid] += 1
    print("Classi:", names)
    print("Box per classe:", dict(zip(names, counts)))
    print("Immagini negative:", negatives, "/", tot_imgs)

# Esegui:
stats_labels(players_root, NEW_NAMES)


Classi: ['player', 'referee', 'rim']
Box per classe: {'player': 37106, 'referee': 9878, 'rim': 3909}
Immagini negative: 0 / 4084


# Passo 9 — Config comuni di training

In [21]:
# Config comuni per L4
COMMON = dict(
    device=0,
    workers=2,
    cache="ram",
    project=f"{RUNS_DIR}/detect",
    exist_ok=True,
    save=True,
    save_period=0, # nessun checkpoint numerato extra (puoi anche omettere questa riga)
    patience=35
)

# Suggerimenti recall palla: imgsz alto, batch auto con accumulate
BALL_TRAIN_CFG = dict(
    model="yolov8m.pt",
    data=str(BALL_DATA_YAML),   # <-- dataset derivato ball-only
    imgsz=1280,
    batch=-1,
    accumulate=4,
    epochs=150,
    patience=30,
    optimizer="auto",
    lr0=0.003, lrf=0.2, momentum=0.937, weight_decay=0.0005,
    warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1,
    multi_scale=False, rect=True,
    mosaic=0.6, close_mosaic=20,
    mixup=0.0, copy_paste=0.0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    translate=0.05, scale=0.4,
    fliplr=0.2, flipud=0.0,
    degrees=0.0, shear=0.0, perspective=0.0,
    max_det=300,
    name="ball_only_1280_yv8m",
    box=8.0, cls=0.5, dfl=1.5,   # opzionale: pesi loss
)


GENERAL_TRAIN_CFG = dict(
    model="yolov8s.pt",           # se vuoi più AP, prova 'yolov8m.pt'
    data=str(DATA_YAML),          # <-- dataset ORIGINALE con tutte le classi
    imgsz=960,                    # 640–960: 960 è un buon compromesso
    batch=-1,
    epochs=120,
    patience=30,
    optimizer="auto",
    lr0=0.003, lrf=0.2, momentum=0.937, weight_decay=0.0005,
    warmup_epochs=3.0, warmup_momentum=0.8, warmup_bias_lr=0.1,
    multi_scale=True, rect=True,
    mosaic=1.0, close_mosaic=10,
    mixup=0.1,
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.3,
    translate=0.05, scale=0.3,
    fliplr=0.5, flipud=0.0,
    degrees=0.0, shear=0.0, perspective=0.0,
    max_det=300,
    name="general_fullclasses_960_yv8s"
    # Se vuoi, puoi aggiungere: box/cls/dfl
)


# (Facoltativo) pesi loss
# BALL_TRAIN_CFG.update(dict(box=8.0, cls=0.5, dfl=1.5))
# GENERAL_TRAIN_CFG.update(dict(box=7.0, cls=0.6, dfl=1.5))  # abilita se serve


# Passo 10 — Avvia training Ball-Only

In [35]:
from pathlib import Path
from ultralytics import YOLO

# 1) Safety check
assert Path(BALL_DATA_YAML).exists(), f"DATA YAML mancante: {BALL_DATA_YAML}"

# 2) Ricostruisci cfg da zero (senza 'accumulate')
cfg = {
    **COMMON,  # contiene già: save=True, save_period=0, project=..., patience=35, ecc.
    **{k: v for k, v in BALL_TRAIN_CFG.items() if k not in ("model", "accumulate", "batch")}
}
cfg["batch"] = -1          # autobatch
cfg["seed"]  = 0
cfg["plots"] = True

print("Estratto cfg:", {k: cfg[k] for k in ("data","imgsz","batch","epochs","seed")})
print("Tipo batch:", type(cfg["batch"]))
print("save:", cfg.get("save"), "| save_period:", cfg.get("save_period"))

# 3) Avvia training (senza ripassare save/save_period)
ball_model = YOLO(BALL_TRAIN_CFG["model"])
res_ball = ball_model.train(**cfg)
print("Fine training Ball-only.")

Estratto cfg: {'data': '/content/datasets_ball_only/data_ball_only.yaml', 'imgsz': 1280, 'batch': -1, 'epochs': 150, 'seed': 0}
Tipo batch: <class 'int'>
save: True | save_period: 0
Ultralytics 8.3.192 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=8.0, cache=ram, cfg=None, classes=None, close_mosaic=20, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets_ball_only/data_ball_only.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.2, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.003, lrf=0.2, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentu

# Step 11 — Avvia training Players/Referee/Rim

In [ ]:
from pathlib import Path
from ultralytics import YOLO

# Safety check
assert Path(DATA_YAML).exists(), f"DATA YAML mancante: {DATA_YAML}"

# Costruisci cfg (niente duplicati di save/save_period: sono in COMMON)
gen_cfg = {
    **COMMON,
    **{k: v for k, v in GENERAL_TRAIN_CFG.items() if k != "model"}
}
# per sicurezza, rimuovi chiavi non supportate se in futuro compaiono
gen_cfg.pop("accumulate", None)

print("Training GENERALE con config:", {k: gen_cfg[k] for k in ("data","imgsz","batch","epochs","patience")})
gen_model = YOLO(GENERAL_TRAIN_CFG["model"])
res_gen = gen_model.train(**gen_cfg)
print("Fine training Generale.")


Training GENERALE con config: {'data': '/content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/data_sanitized.yaml', 'imgsz': 960, 'batch': -1, 'epochs': 120, 'patience': 30}
Ultralytics 8.3.192 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/data_sanitized.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.6, hsv_v=0.3, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=No

# Step 12 — Valutazione e report rapidi

In [ ]:
from pathlib import Path
import pandas as pd
import os

def last_run_dir(name_substr: str):
    runs = sorted(Path(RUNS_DIR) / "detect".glob(f"**/*{name_substr}*"), key=os.path.getmtime)
    return runs[-1] if runs else None

ball_run = last_run_dir(BALL_TRAIN_CFG["name"])
players_run = last_run_dir(PLAYERS_TRAIN_CFG["name"])
print("Ultima run ball:", ball_run)
print("Ultima run players:", players_run)

def read_results_csv(run_dir: Path):
    csv = run_dir / "results.csv"
    return pd.read_csv(csv) if csv.exists() else None

def show_tail(df):
    if df is not None:
        display(df.tail(3))
    else:
        print("Nessun results.csv")

print("\n=== BALL-ONLY metrics ===")
show_tail(read_results_csv(ball_run))
print("\n=== PLAYERS metrics ===")
show_tail(read_results_csv(players_run))

# PR curve e Confusion Matrix sono salvati nei sottopacchetti 'val'
print("\nFigure tipiche salvate in:", (ball_run/"val").as_posix(), "e", (players_run/"val").as_posix())


# (Nuovo) Riprendere il training dopo un'interruzione
Esegui la cella seguente **solo** se la sessione si è interrotta: riprende dall'ultimo `last.pt` salvato su Drive.


In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

def resume_latest(exp_name: str):
    base = Path(RUNS_DIR) / 'detect'
    candidates = [p for p in base.glob(f'**/*{exp_name}*') if (p / 'weights/last.pt').exists()]
    assert candidates, f'Nessuna run trovata per {exp_name} in {base}'
    rd = sorted(candidates, key=os.path.getmtime)[-1]
    last = rd / 'weights/last.pt'
    print('Riprendo da:', last)
    m = YOLO(str(last))
    return m.train(resume=True)


# Esempi:
# resume_latest(BALL_TRAIN_CFG['name'])
resume_latest(GENERAL_TRAIN_CFG['name'])

Riprendo da: /content/drive/MyDrive/basketball_analysis/basketball_analysis/last_runs/detect/general_fullclasses_960_yv8s/weights/last.pt
Ultralytics 8.3.192 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=19, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/basketball_analysis/basketball_analysis/basketball-player-detection-2-1/data_sanitized.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.6, hsv_v=0.3, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.003, lrf=0.2, mask_ratio=4, max_det=300, mixup=0.0, mode=train, mod

 # Step 13 — Percorsi dei pesi finali (per integrazione nel tracker)

In [ ]:
def best_pt_of(run_dir: Path):
    cand = run_dir / "weights" / "best.pt"
    return cand if cand.exists() else None

ball_best = best_pt_of(ball_run)
players_best = best_pt_of(players_run)
print("BALL_DETECTOR_PATH:", ball_best)
print("PLAYER_DETECTOR_PATH:", players_best)

# Copia facoltativa in una cartella fissa su Drive
EXPORT_DIR = Path(PROJECT_PATH) / "exports_models"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
if ball_best:
    shutil.copy2(ball_best, EXPORT_DIR / f"{BALL_TRAIN_CFG['name']}_best.pt")
if players_best:
    shutil.copy2(players_best, EXPORT_DIR / f"{PLAYERS_TRAIN_CFG['name']}_best.pt")
print("Esportati in:", EXPORT_DIR)
!ls -la "$EXPORT_DIR"


# DOPO L'ADDESTRAMENTO:

VERIFICA 1

In [ ]:
START = # TODO: modifica con path modello - dopo addestramento

import torch, os
import ultralytics
print("torch:", torch.__version__, "| ultralytics:", ultralytics.__version__)

ckpt = torch.load(START, map_location="cpu", weights_only=False)  # <-- chiave
print("Chiavi:", list(ckpt.keys())[:10])
print("Optimizer presente?", ("optimizer" in ckpt) and (ckpt["optimizer"] is not None))
print("Epoch salvata:", ckpt.get("epoch"))
print("Versione Ultralytics nel ckpt:", ckpt.get("version"))
print("Dimensione file (MB):", os.path.getsize(START)/1024/1024)

VERIFICA 2

In [ ]:
import torch, os
ckpt = torch.load(START, map_location="cpu", weights_only=False)
print("optimizer?", ckpt.get("optimizer") is not None, "| epoch:", ckpt.get("epoch"))
print("sizeMB:", os.path.getsize(START)/1024/1024)

VERIFICA 3

In [ ]:
import os, torch
ckpt = torch.load(START, map_location="cpu", weights_only=False)
data_yaml_saved = ckpt.get("train_args", {}).get("data")
print("data.yaml salvato:", data_yaml_saved, "| esiste?", os.path.exists(str(data_yaml_saved)))

# DOWNLOAD FINALE - SE NECESSARIO - DOVREBBE SALVARE IN AUTOMATICO SU DRIVE

In [ ]:
from google.colab import files
from pathlib import Path
# Usa last_run_dir (definita sopra). In caso contrario, trova il run più recente in Drive.
try:
    run_dir = last_run_dir('unified_detector')
except Exception:
    from glob import glob
    cand = sorted((Path(RUNS_DIR)/'detect').glob('**/*'), key=lambda p: p.stat().st_mtime if p.exists() else 0)
    run_dir = cand[-1] if cand else None
assert run_dir is not None, 'Nessuna run trovata sotto RUNS_DIR/detect'
best = Path(run_dir) / 'weights/best.pt'
print(f'Download del modello migliore da: {best}')
files.download(str(best))


Download del modello migliore da: runs/detect/unified_detector_v1.1/weights/best.pt


FileNotFoundError: Cannot find file: runs/detect/unified_detector_v1.1/weights/best.pt

Dopo che la Cella 5 ha finito, il tuo nuovo modello sarà stato salvato. Puoi verificare il percorso esatto e usarlo. Il file che ti interessa sarà in:
  `runs/pose/finetune_court_detector/weights/best.pt`

Questo è il file che dovrai scaricare da Google Drive e usare nella tua applicazione principale.

Questo nuovo notebook è più pulito, più robusto e, soprattutto, esegue correttamente il fine-tuning come avevamo pianificato.